In [1]:
from datetime import datetime

import numpy as np
import pandas as pd

from src.databases import PostgresCRUD

In [2]:
def check_exclude_matching(df_orin: pd.DataFrame) -> pd.DataFrame:
    """집계 제외 Recipe 선정

    Args:
        df_orin (pd.DataFrame): Recipe 리스트

    Returns:
        pd.DataFrame: 수정된 Recipe 리스트
    """
    # STD_RECIPE = PostgresCRUD.select_query_by_list("select recipe from std_recipe",)
    test_device = PostgresCRUD.select_query_by_list("select device from test_device",)
    results = []
    for row in df_orin.itertuples(index=False):
        tech_prod, tech, prod_group, level, layer, mode, device, parse, model, master = (
            row.tech_prod, row.tech, row.prod_group, row.level, row.layer,
            row.mode, row.device, row.parse_status, row.model, row.master_key
        )
        # rcp_tag = layer[4:]
        if (device in test_device) or (prod_group == "TBD"):
            is_exc, code = True, "EXC_TEST"
        elif not isinstance(device, str) or len(device) != 4:
            is_exc, code = True, "EXC_DEVICE"
        elif not isinstance(layer, str) or len(layer) != 8:
            is_exc, code = True, "EXC_LAYER"
        # elif not (any(rcp.endswith(rcp_tag) for rcp in STD_RECIPE) or (layer in STD_RECIPE)):
        #     is_exc, code = True, "EXC_RECIPE"
        elif prod_group == "IGBT":
            is_exc, code = True, "EXC_IGBT"
        elif parse == "failed":
            is_exc, code = True, "EXC_PARSE"
        elif master is None:
            is_exc, code = True, "EXC_MASTER"
        elif model in ("KLA2139", "KLA2360", "KLA2365", "KLA2367")\
            and not all([tech_prod, tech, prod_group, level, layer, mode]):
            is_exc, code = True, "EXC_MISS"
        elif model == "PUMA" and not all([tech_prod, tech, prod_group, level, layer]):
            is_exc, code = True, "EXC_MISS"
        else:
            is_exc, code = False, None            
        results.append((is_exc, code))

    # 결과를 DataFrame에 바로 컬럼으로 할당 (merge 비용 절감)
    df_update = df_orin.copy()
    df_update["exclude_matching"] = [r[0] for r in results]
    df_update["exclude_code"] = [r[1] for r in results]
    return df_update

def check_para_matching(value: str, master: str):
    """ Parameter 매칭 확인 """
    if (master is None) | (value is None):
        return None
    if (isinstance(master, float)) and np.isnan(master):
        return None
    if '/' in master:
        return value in master.split('/')
    if '~' in master:
        lower, upper = map(float, master.split('~'))
        return lower<=float(value)<=upper
    if value is None:
        return False
    return str.upper(value) == str.upper(master)

In [ ]:
# [장비 단위]
# Master 기준정보:
# model, tech_prod, tech, prod_group, level, layer, mode: KLA2139, KLA2360, KLA2365, KLA2367
# model, tech_prod, tech, prod_group, level, layer: PUMA

# 1. Master Key 업데이트
query = """
UPDATE dm_recipe_info AS dri
SET master_key = dmri.master_key
FROM dm_master_recipe_info AS dmri
WHERE dri.model = dmri.model
    AND dri.tech_prod = dmri.tech_prod
    AND dri.tech = dmri.tech
    AND dri.prod_group = dmri.prod_group
    AND dri.level = dmri.level
    AND dri.layer = dmri.layer
    AND (
        dri.model = 'PUMA'
        OR dri.mode = dmri.mode
    )
"""
PostgresCRUD.execute(query)

# 2. dm_recipe 불러오기
df_recipe = PostgresCRUD.select_query("SELECT * FROM dm_recipe_info", )

# 3. 집계 제외 Recipe 업데이트
df_recipe = check_exclude_matching(df_recipe)
PostgresCRUD.upsert("dm_recipe_info", "dm_recipe_info_pkey", df_recipe[["recipe_key", "exclude_matching", "exclude_code"]])

# 4. Recipe의 Parameter별 Master Value 불러오기
query = """
SELECT 
	dri.model,
	dri.equip,
	dri.recipe_key,
	dri.master_key,
	dri.recipe, 
	drp.para_nm,
	drp.para_value AS value,
	dmrp.para_value AS master_value,
	drp.matching AS para_matching
FROM dm_recipe_info AS dri
LEFT JOIN dm_recipe_para AS drp
ON (dri.recipe_key = drp.recipe_key)
LEFT JOIN dm_master_recipe_para AS dmrp
ON (
	dri.master_key = dmrp.master_key
		AND drp.para_nm = dmrp.para_nm
)
WHERE dri.is_remove = false
	AND dri.exclude_matching = false
	AND dri.master_key is not null
	AND drp.para_nm != 'mode'
	AND drp.para_value is not null
	AND dmrp.para_value is not null
ORDER BY dri.recipe_key, drp.para_nm
"""
df_para_matching = PostgresCRUD.select_query(query, )

# 5. dm_recipe_para Matching 집계 및 업데이트 - Parameter 단위 Matching
result = [check_para_matching(value, master) for value, master
          in zip(df_para_matching["value"], df_para_matching["master_value"])]
df_para_matching["para_matching"] = result
PostgresCRUD.upsert("dm_recipe_para", "dm_recipe_para_pk", df_para_matching[["recipe_key", "para_nm", "para_matching"]].rename(columns={"para_matching":"matching"}))

# 6. dm_recipe_info Matching 집계 및 업데이트 - Recipe 단위 Matching
rcp_matching_info = {
    rcp_key:all(rcp_data["para_matching"])
    for rcp_key, rcp_data
    in df_para_matching.groupby("recipe_key")
}
df_para_matching["matching"] = df_para_matching["recipe_key"].map(rcp_matching_info)
PostgresCRUD.upsert("dm_recipe_info", "dm_recipe_info_pkey", df_para_matching[["recipe_key","matching"]].drop_duplicates(subset=["recipe_key"], keep="first"))

# 7. 장비 단위 Matching Rate 집계 및 업데이트
matching_result = []
agg_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
for (equip, model), eq_data in df_recipe.groupby(["equip", "model"]):
    eq_matching_data = df_para_matching[df_para_matching["equip"]==equip].drop_duplicates(subset=["recipe_key"], keep="first")
    matching_info = eq_matching_data["matching"].value_counts()
    agg_recipe = len(eq_matching_data)
    matching_recipe = int(matching_info.get(True, 0))
    matching_rate = matching_recipe / agg_recipe if agg_recipe > 0 else 0
    matching_result.append({
        "model": model,
        "equip": equip,
        "recipe": len(eq_data),
        "matching": matching_recipe,
        "non_matching": int(matching_info.get(False, 0)),
        "exclude_matching": len(eq_data[eq_data["exclude_matching"]]),
        "matching_rate": matching_rate,
        "aggregation_time": agg_time
    })
df_matching_rate = pd.DataFrame(matching_result)
total_info = []
for model, model_matching_data in df_matching_rate.groupby("model"):
    matching = int(model_matching_data["matching"].sum())
    non_matching = int(model_matching_data["non_matching"].sum())
    matching_rate = matching / (matching+non_matching) if (matching+non_matching)>0 else 0
    total_info.append({
        "model": model,
        "equip": "TOTAL",
        "recipe": int(model_matching_data["recipe"].sum()),
        "matching": matching,
        "non_matching": non_matching,
        "exclude_matching": int(model_matching_data["exclude_matching"].sum()),
        "matching_rate": matching_rate,
        "aggregation_time": agg_time
    })
df_matching_rate = pd.concat([df_matching_rate, pd.DataFrame(total_info)], ignore_index=True)
PostgresCRUD.upsert("dm_matching_rate", "dm_matching_rate_pkey", df_matching_rate)

In [12]:
df_para_matching.head()

,model,equip,recipe_key,master_key,recipe,para_nm,value,master_value,para_matching,matching
0,KLA2139,SMDI01,SMDI01-A003_1565PAD3,KLA2139_00180,A003_1565PAD3,edge,Include,INCLUDE,True,True
1,KLA2139,SMDI01,SMDI01-A003_1565PAD3,KLA2139_00180,A003_1565PAD3,filter,F0,F0,True,True
2,KLA2139,SMDI01,SMDI01-A003_1565PAD3,KLA2139_00180,A003_1565PAD3,focus_offset,-0.3048,-1.1~0.4,True,True
3,KLA2139,SMDI01,SMDI01-A003_1565PAD3,KLA2139_00180,A003_1565PAD3,pixel,0.62,0.62,True,True
4,KLA2139,SMDI01,SMDI01-A003_1565PAD3,KLA2139_00180,A003_1565PAD3,threshold,16,16~24,True,True


In [21]:
df_para_matching[["recipe_key","matching"]].drop_duplicates(subset=["recipe_key"], keep="first")

,recipe_key,matching
0,SMDI01-A003_1565PAD3,True
5,SMDI01-A003_290662UM,True
10,SMDI01-A003_615562UM,True
25,SMDI01-A003_616562UM,True
40,SMDI01-A003_752562UM,True
...,...,...
712206,SMDI06-ZY63_5085COSI,True
712211,SMDI06-ZY63_615562UM,True
712226,SMDI06-ZY63_644562UM,True
712241,SMDI06-ZY63_665562UM,True


In [11]:
df_matching_rate

,model,equip,recipe,matching,non_matching,exclude_matching,matching_rate,aggregation_time
0,KLA2139,SMDI01,21931,21667,146,118,0.993307,2026-11-05 11:34:04
1,KLA2139,SMDI02,20713,20546,109,58,0.994723,2026-11-05 11:34:04
2,KLA2139,SMDI03,21095,20934,81,80,0.996146,2026-11-05 11:34:04
3,KLA2139,SMDI06,21825,21396,328,101,0.984901,2026-11-05 11:34:04
4,KLA2365,SMDI08,5058,0,0,5058,0.000000,2026-11-05 11:34:04
5,PUMA,SMPW06,10019,0,0,10019,0.000000,2026-11-05 11:34:04
6,KLA2139,TOTAL,85564,84543,664,357,0.992207,2026-11-05 11:34:04
7,KLA2365,TOTAL,5058,0,0,5058,0.000000,2026-11-05 11:34:04
8,PUMA,TOTAL,10019,0,0,10019,0.000000,2026-11-05 11:34:04


In [4]:
# 장비 별 master_key 매칭 안하는 Recipe 확인
df_recipe[df_recipe["master_key"].isnull()]["equip"].value_counts()

equip
SMPW06    10019
SMDI08     5058
SMDI01       91
SMDI06       89
SMDI03       62
SMDI02       44
Name: count, dtype: int64

In [5]:
# 집계 제외 코드 확인
df_recipe["exclude_code"].value_counts()

exclude_code
EXC_MASTER    14962
EXC_TEST        286
EXC_PARSE       113
EXC_LAYER        43
EXC_IGBT         30
Name: count, dtype: int64

In [6]:
# 장비 별 집계 제외 Recipe 확인
df_recipe[df_recipe["exclude_matching"]]["equip"].value_counts()

equip
SMPW06    10019
SMDI08     5058
SMDI01      118
SMDI06      101
SMDI03       80
SMDI02       58
Name: count, dtype: int64